In [2]:
# ==========================================
# Notebook 6: Train tune evaluate
# ==========================================
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc, 
    f1_score, recall_score, precision_score, classification_report, confusion_matrix
)
from xgboost import XGBClassifier

# Configuration
os.makedirs("artifacts/models", exist_ok=True)
os.makedirs("artifacts/charts", exist_ok=True)

# 1. Chargement des datasets prétraités
train_df = pd.read_parquet("artifacts/05_train_features.parquet")
val_df = pd.read_parquet("artifacts/05_val_features.parquet")
test_df = pd.read_parquet("artifacts/05_test_features.parquet")

target_col = 'is_late'

X_train, y_train = train_df.drop(columns=[target_col]), train_df[target_col]
X_val, y_val = val_df.drop(columns=[target_col]), val_df[target_col]
X_test, y_test = test_df.drop(columns=[target_col]), test_df[target_col]

print(f"Train features: {X_train.shape}, Val features: {X_val.shape}, Test features: {X_test.shape}")

Train features: (67534, 65), Val features: (14472, 65), Test features: (14472, 65)


In [3]:
# Fonction d'évaluation adaptée aux classes déséquilibrées (PR-AUC principal)
def evaluate_model(model, X, y, name="Model"):
    y_pred_proba = model.predict_proba(X)[:, 1]
    y_pred = model.predict(X)
    
    # Calcul PR-AUC (Precision-Recall AUC)
    precision, recall, _ = precision_recall_curve(y, y_pred_proba)
    pr_auc = auc(recall, precision)
    roc_auc = roc_auc_score(y, y_pred_proba)
    f1 = f1_score(y, y_pred)
    rec = recall_score(y, y_pred)
    prec = precision_score(y, y_pred, zero_division=0)
    
    return {
        'Model': name,
        'PR-AUC': pr_auc,
        'ROC-AUC': roc_auc,
        'F1-Score': f1,
        'Recall': rec,
        'Precision': prec
    }

results = []

# 1. Baseline Aléatoire / Naïve (Dummy)
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
results.append(evaluate_model(dummy, X_val, y_val, name="Baseline (Dummy)"))

# 2. Baseline Régression Logistique (avec gestion du poids des classes)
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
results.append(evaluate_model(log_reg, X_val, y_val, name="Logistic Regression"))

# 3. Random Forest (Class Weight Balanced)
rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
results.append(evaluate_model(rf, X_val, y_val, name="Random Forest"))

# 4. XGBoost (Tuning scale_pos_weight pour le déséquilibre)
scale_pos = (len(y_train) - sum(y_train)) / sum(y_train)
xgb = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.05, scale_pos_weight=scale_pos, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
results.append(evaluate_model(xgb, X_val, y_val, name="XGBoost (Tuned)"))

# Tableau comparatif sur la Validation
val_results_df = pd.DataFrame(results).sort_values(by='PR-AUC', ascending=False)
print("=== RÉSULTATS SUR LE VALIDATION SET ===")
print(val_results_df.to_string(index=False))

=== RÉSULTATS SUR LE VALIDATION SET ===
              Model   PR-AUC  ROC-AUC  F1-Score   Recall  Precision
   Baseline (Dummy) 0.526707 0.500000  0.000000 0.000000   0.000000
Logistic Regression 0.088949 0.644776  0.147607 0.548512   0.085278
      Random Forest 0.075980 0.605251  0.140975 0.276843   0.094565
    XGBoost (Tuned) 0.073295 0.562362  0.110415 0.125485   0.098577


In [6]:
# Sélection du meilleur modèle basé sur le PR-AUC de Validation
best_model = log_reg  
best_model_name = "Logistic Regression"

print(f"\nÉvaluation unique du meilleur modèle ({best_model_name}) sur le TEST SET...")

# Métriques sur le Test Set
test_metrics = evaluate_model(best_model, X_test, y_test, name=f"{best_model_name} (Test)")
test_results_df = pd.DataFrame([test_metrics])

print("\n=== RÉSULTATS FINAUX SUR LE TEST SET ===")
print(test_results_df.to_string(index=False))

# Matrice de confusion sur le Test Set
y_test_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Dans les temps', 'En retard'], yticklabels=['Dans les temps', 'En retard'])
plt.ylabel('Réalité')
plt.xlabel('Prédiction')
plt.title(f'Matrice de Confusion (Test Set) - {best_model_name}')
plt.savefig("artifacts/charts/test_confusion_matrix.png", bbox_inches='tight')
plt.close()


Évaluation unique du meilleur modèle (Logistic Regression) sur le TEST SET...

=== RÉSULTATS FINAUX SUR LE TEST SET ===
                     Model   PR-AUC  ROC-AUC  F1-Score   Recall  Precision
Logistic Regression (Test) 0.062983 0.424053  0.088633 0.249739   0.053877


In [8]:
# 1. Sauvegarde du modèle entraîné
joblib.dump(best_model, "artifacts/models/best_model.joblib")

# 2. Rédaction du résumé des résultats
summary_text = f"""
==================================================
        MODEL TRAINING & EVALUATION SUMMARY
==================================================

1. Choix de la métrique :
   - Problème déséquilibré (~6-8% de retards).
   - PR-AUC (Precision-Recall AUC) et F1-Score ont été privilégiés par rapport à l'Accuracy.

2. Performances sur le Validation Set :
{val_results_df.to_string(index=False)}

3. Évaluation Finale sur le Test Set ({best_model_name}) :
   - PR-AUC   : {test_metrics['PR-AUC']:.4f}
   - ROC-AUC  : {test_metrics['ROC-AUC']:.4f}
   - F1-Score : {test_metrics['F1-Score']:.4f}
   - Recall   : {test_metrics['Recall']:.4f}
   - Precision: {test_metrics['Precision']:.4f}

4. Conclusion & Prochaines étapes MLOps :
   - Le modèle final à été sauvegardé sous 'artifacts/models/best_model.joblib'.
   - Prêt pour l'étape de déploiement d'API d'inférence (Notebook 07).
==================================================
"""

with open("artifacts/model_results_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)
print("\n✅ Modèle sauvegardé dans 'artifacts/models/best_model.joblib'")
print("📝 Synthèse sauvegardée dans 'artifacts/model_results_summary.txt'")


        MODEL TRAINING & EVALUATION SUMMARY

1. Choix de la métrique :
   - Problème déséquilibré (~6-8% de retards).
   - PR-AUC (Precision-Recall AUC) et F1-Score ont été privilégiés par rapport à l'Accuracy.

2. Performances sur le Validation Set :
              Model   PR-AUC  ROC-AUC  F1-Score   Recall  Precision
   Baseline (Dummy) 0.526707 0.500000  0.000000 0.000000   0.000000
Logistic Regression 0.088949 0.644776  0.147607 0.548512   0.085278
      Random Forest 0.075980 0.605251  0.140975 0.276843   0.094565
    XGBoost (Tuned) 0.073295 0.562362  0.110415 0.125485   0.098577

3. Évaluation Finale sur le Test Set (Logistic Regression) :
   - PR-AUC   : 0.0630
   - ROC-AUC  : 0.4241
   - F1-Score : 0.0886
   - Recall   : 0.2497
   - Precision: 0.0539

4. Conclusion & Prochaines étapes MLOps :
   - Le modèle final à été sauvegardé sous 'artifacts/models/best_model.joblib'.
   - Prêt pour l'étape de déploiement d'API d'inférence (Notebook 07).


✅ Modèle sauvegardé dans 'artifac